# Deduplication and the latest record per entity

Practice notebook for the datafromzero SQL lesson **Deduplication and the latest record per entity**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## The practice table

cafe-sales has no repeated keys, so this lesson also uses a small table typed by hand: `customer_updates`, ten
invented rows of a loyalty-app export with repeats. Run this cell once. The table lives only in this session. A later
lesson, *Creating tables and constraints*, explains how tables are made.

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE customer_updates AS
SELECT * FROM (VALUES
  (1,  301, 'bronze', 'Oakland',  TIMESTAMP '2026-03-02 09:15:00'),
  (2,  301, 'silver', 'Oakland',  TIMESTAMP '2026-05-10 14:02:00'),
  (3,  302, 'bronze', 'Berkeley', TIMESTAMP '2026-03-05 10:30:00'),
  (4,  302, 'bronze', 'Berkeley', TIMESTAMP '2026-03-05 10:30:00'),
  (5,  303, 'silver', 'Alameda',  TIMESTAMP '2026-04-01 08:00:00'),
  (6,  303, 'gold',   'Alameda',  TIMESTAMP '2026-04-01 08:00:00'),
  (7,  304, 'bronze', 'Oakland',  TIMESTAMP '2026-04-20 16:45:00'),
  (8,  305, 'bronze', 'Berkeley', TIMESTAMP '2026-02-11 11:11:00'),
  (9,  305, 'silver', 'Berkeley', TIMESTAMP '2026-06-01 09:30:00'),
  (10, 305, 'silver', 'Berkeley', TIMESTAMP '2026-06-01 09:30:00')
) AS t(update_id, customer_id, loyalty_tier, city, updated_at)
""")
print("customer_updates is ready")

## Worked example

### Step 1. A table with no repeated keys

In [ ]:
con.sql("""
SELECT order_id, COUNT(*) AS copies
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1
""").show()

### Step 2. The practice table

In [ ]:
con.sql("""
SELECT * FROM customer_updates ORDER BY update_id
""").show()

### Step 3. Find the repeated customer ids

In [ ]:
con.sql("""
SELECT customer_id, COUNT(*) AS records
FROM customer_updates
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY customer_id
""").show()

### Step 4. Why DISTINCT is not the fix

In [ ]:
con.sql("""
SELECT DISTINCT customer_id, loyalty_tier, city, updated_at
FROM customer_updates
ORDER BY customer_id, updated_at
""").show()

### Step 5. State the survivor rule and number the rows

In [ ]:
con.sql("""
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY customer_id
                          ORDER BY updated_at DESC, update_id DESC) AS rn
FROM customer_updates
ORDER BY customer_id, rn
""").show()

### Step 6. Keep one row per customer

In [ ]:
con.sql("""
WITH numbered AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY customer_id
                            ORDER BY updated_at DESC, update_id DESC) AS rn
  FROM customer_updates
)
SELECT update_id, customer_id, loyalty_tier, city, updated_at
FROM numbered
WHERE rn = 1
ORDER BY customer_id
""").show()

In [ ]:
con.sql("""
SELECT update_id, customer_id, loyalty_tier, city, updated_at
FROM customer_updates
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id
                           ORDER BY updated_at DESC, update_id DESC) = 1
ORDER BY customer_id
""").show()

### Step 7. Verify

In [ ]:
con.sql("""
WITH numbered AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY customer_id
                            ORDER BY updated_at DESC, update_id DESC) AS rn
  FROM customer_updates
),
latest AS (
  SELECT * FROM numbered WHERE rn = 1
)
SELECT COUNT(*) AS rows_kept, COUNT(DISTINCT customer_id) AS customers
FROM latest
""").show()

## Exercise 1. Each customer's first record

Keep the first record of each customer instead of the latest: the earliest `updated_at`, and the lower `update_id` on
a tie. The `numbered` stage is started for you.

In [ ]:
con.sql("""
WITH numbered AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY customer_id
                            ORDER BY ...) AS rn
  FROM customer_updates
)
SELECT update_id, customer_id, loyalty_tier, city, updated_at
FROM numbered
""").show()

## Exercise 2. Each customer's latest completed order

Before you run anything, write one sentence in the first cell: how many rows do you expect? cafe-sales has 150
customers, but think about who has no completed order. Then write the query on `orders`, with the columns
`customer_id`, `order_id` and `ordered_at`.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague keeps the latest record of each customer. The query returns 5 rows, one for each customer, so the row
count looks right. Look at customer 303, find the flaw and write the fix.

In [ ]:
con.sql("""
WITH numbered AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY customer_id
                            ORDER BY updated_at DESC) AS rn
  FROM customer_updates
)
SELECT update_id, customer_id, loyalty_tier, updated_at
FROM numbered
WHERE rn = 1
ORDER BY customer_id
""").show()

In [ ]:
# Your corrected query: